# Integrating Unity Catalog functions as tools to Agents
Enables AI agents to perform complex data retrieval tasks by executing SQL functions defined in the Unity Catalog.

## Step1: Unity Catalog function definition
1. Catalog & Schema Setup: Functions are created within a structured Unity Catalog (agents) and schema (main)
2. Function creation: Use SQL to define functions
3. Metadata importance: Include clear comments and descriptions for each function. This metadata is essential for the AI agent to understand when to invoke specific tools. 
4. Retrieval logic: Implement SQL queries within the function to select data from tables

In [0]:

%sql 
create catalog if not exists agents;
create schema if not exists agents.main;

In [0]:
%sql
CREATE OR REPLACE FUNCTION agents.main.get_customer_by_email(email_input STRING COMMENT 'customer email used to retrieve customer information')
RETURNS TABLE (
    customerID BIGINT,
    first_name STRING,
    last_name STRING,
    email_address STRING,
    phone_number STRING,
    address STRING,
    city STRING,
    state STRING,
    postal_zip_code STRING,
    country STRING,
    continent STRING,
    gender STRING
)
COMMENT 'Returns the customer record matching the provided email address. Includes its customerID, first_name, last_name and more.'
RETURN (
    SELECT * FROM samples.bakehouse.sales_customers
    WHERE email_address = email_input
    LIMIT 1
);
     


In [0]:
%sql SELECT * FROM agents.main.get_customer_by_email('brittanyramos@example.org');

## Step 2: Integration with LangGraph
1. Import UCFunctionToolKit to bridge the gap between UC and agent framework
2. Provide the agent with the relevant function names. Using a wildcard configuraion allows for the inclusion of all functions within a specific schema
3. Bind the UC tools to the llm to make them available during the agent's decision-making process

In [0]:
from dotenv import load_dotenv
import os
import random
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import MessagesState
import mlflow
mlflow.langchain.autolog()

In [0]:
class State(MessagesState):
    pass

In [0]:
llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")
uc_tool_names  = ("agents.main.*",)
tools  = UCFunctionToolkit(function_names=list(uc_tool_names)).tools
llm_with_tools = llm.bind_tools(tools)

In [0]:
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Step 3: Agent Execution and Reasoning
1. Agent evaluates user queries against the function metadata to determine which tool call is necessary
2. If a query requires data, the agent automatically calls the corresponding function
3. Once the function returns the results (in CSV format), the agent formats the output into a readable response for the user
4. Agents are capable of iterating - if one tool call fails to provide the necessary information, they can attempt to use other availablet tools to complete the task

In [0]:
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))


builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("Can you tell me more info about brittanyramos@example.org?")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")

In [0]:

result = graph.invoke({"messages": [HumanMessage("Which table are you using to get the user details")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")

In [0]:
result = graph.invoke({"messages": [HumanMessage("Tell me more about the user table")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")